# Lab 3 · NumPy trên dữ liệu thật về giá phòng của Santiago

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · bài 3**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo bài 3 dùng các mảng nhỏ tự tạo để học cú pháp. Trong lab này, bạn áp dụng
NumPy lên **17.688 mức giá thật** của Santiago — và lên ma trận review 4 năm × 12 tháng.

## Cách làm việc trong buổi lab

- Bài tập được chia thành từng bước; mỗi bước có ô `TODO` và phần kiểm tra `assert`.
  Hoàn thành toàn bộ `assert` nghĩa là kết quả đáp ứng yêu cầu.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🚫 đóng ở giờ lý thuyết đánh giá các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn ✅ mở: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Nếu chưa giải quyết được một bước sau 3 phút, hãy gọi giảng viên thực hành đến hỗ trợ.

## Mục tiêu

Sau buổi lab, bạn sẽ:

1. Đưa được một cột dữ liệu thật vào NumPy và xử lý NaN chủ động.
2. Dùng mặt nạ bool để đếm, tính tỷ lệ, lọc và ghép điều kiện.
3. Dùng `np.where`, phân vị để gắn nhãn và tìm ngưỡng ngoại lai.
4. Đọc đúng `axis` và `argmax` trên ma trận 2 chiều thật.

## Phần 0 · Khởi động (~10 phút)

Ba thao tác lõi của bài giảng, trên mảng nhỏ.

In [31]:
import numpy as np

# W1 — tạo mảng và xem thuộc tính
a = np.array([4, 8, 15, 16, 23, 42])

# TODO: điền 2 thuộc tính của a
so_phan_tu = a.size        # dùng a.shape hoặc a.size
kieu = a.dtype             # dùng a.dtype (giữ nguyên object dtype, không đổi chuỗi)

# --- Ô kiểm tra ---
assert so_phan_tu == 6 and str(kieu) == "int64"
print("W1 ổn:", a.shape, a.dtype)

W1 ổn: (6,) int64


In [32]:
# W2 — mặt nạ bool: đếm và tỷ lệ
diem = np.array([7.5, 4.0, 9.0, 5.5, 8.0, 3.0, 6.5, 9.5])

# TODO: đếm số điểm >= 5 và tỷ lệ điểm >= 8 (một biểu thức mỗi dòng)
so_dat = (diem >= 5).sum()
ty_le_gioi = (diem >= 8).mean()

# --- Ô kiểm tra ---
assert so_dat == 6 and ty_le_gioi == 0.375
print(f"W2 ổn: {so_dat} bài đạt, {ty_le_gioi:.0%} giỏi")

W2 ổn: 6 bài đạt, 38% giỏi


In [33]:
# W3 — axis trên ma trận 2x3
M = np.array([[1, 2, 3],
              [4, 5, 6]])

# TODO: tổng theo CỘT (một số cho mỗi cột) và tổng theo HÀNG
tong_cot = np.sum(M,axis = 0)
tong_hang = M.sum(axis = 1)

# --- Ô kiểm tra ---
assert list(tong_cot) == [5, 7, 9] and list(tong_hang) == [6, 15]
print("W3 ổn — axis nào bị gọi tên, chiều đó biến mất.")

W3 ổn — axis nào bị gọi tên, chiều đó biến mất.


## Phần 1 · Giá thật vào NumPy (~40 phút)

Cột giá được lấy bằng một dòng pandas (bài 4 học kỹ), sau đó được xử lý bằng NumPy.

In [34]:
import pandas as pd

URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/visualisations/listings.csv")
gia = pd.read_csv(URL)["price"].to_numpy()

gia.dtype, gia.shape

(dtype('float64'), (18534,))

### Bước 1 · NaN — làm sạch trước khi tính

Cột giá là `float64` và chứa `nan` (các phòng không khai giá — bạn đã gặp chúng ở lab 2).

In [35]:
# TODO: đếm số NaN bằng np.isnan, rồi tạo mảng gia_sach không còn NaN
so_nan = np.isnan(gia).sum()
gia_sach = gia[~np.isnan(gia)]          # gợi ý: mặt nạ ~np.isnan(gia)

# --- Ô kiểm tra ---
assert so_nan == 846 and gia_sach.shape == (17688,)
assert not np.isnan(gia_sach).any()
print(f"Bỏ {so_nan} NaN, còn {gia_sach.size:,} giá hợp lệ.")

Bỏ 846 NaN, còn 17,688 giá hợp lệ.


Vì sao không dùng luôn `np.nanmean` cho mọi thứ? Vì các phép sau (lọc, phần trăm,
nhãn) đều cần mảng không còn NaN. Làm sạch một lần ở đầu giúp các bước sau đơn giản hơn;
đây cũng là lý do pipeline có một bước làm sạch riêng.

### Bước 2 · Câu hỏi thị trường bằng mặt nạ bool

In [36]:
# TODO: trả lời 3 câu bằng mặt nạ bool trên gia_sach
ty_le_tren_100k = (gia_sach > 100_000).mean()     # tỷ lệ phòng giá > 100.000 CLP (mean trên bool)
so_duoi_30k = (gia_sach < 30_000).sum()         # số phòng giá < 30.000 CLP
so_khoang_50_100 = ((50_000 <= gia_sach) & (gia_sach <= 100_000)).sum()     # số phòng 50.000 <= giá <= 100.000 (ghép 2 điều kiện bằng &)

# --- Ô kiểm tra ---
assert round(ty_le_tren_100k, 4) == 0.2088
assert so_duoi_30k == 1660
assert so_khoang_50_100 == (( (gia_sach >= 50_000) & (gia_sach <= 100_000) ).sum())
print(f"{ty_le_tren_100k:.1%} trên 100k · {so_duoi_30k} phòng dưới 30k")

20.9% trên 100k · 1660 phòng dưới 30k


### Bước 3 · np.where — gắn nhãn cho cả mảng

In [37]:
med = np.median(gia_sach)          # 59000.0

# TODO: tạo mảng nhan: "cao" nếu giá > med, ngược lại "pho thong"
nhan = np.where(gia_sach > med, "cao","pho thong")

# --- Ô kiểm tra ---
assert med == 59000.0
assert (nhan == "cao").sum() == 8841
print(f"Trung vị {med:,.0f} CLP — {(nhan == 'cao').sum():,} phòng thuộc nửa 'cao'.")

Trung vị 59,000 CLP — 8,841 phòng thuộc nửa 'cao'.


Để ý: "nửa cao" chỉ có 8.841 phòng — chưa đúng một nửa của 17.688. Vì sao?
Có 4 phòng có giá **đúng bằng** trung vị nên rơi vào nhánh "pho thong". Cần lưu ý rằng
ranh giới thuộc về bên nào là **quyết định của bạn**.

### Bước 4 · Phân vị (percentile) — xác định ngưỡng ngoại lai (outlier)

In [38]:
# TODO: tính P1, P99 của gia_sach (np.percentile) và đếm số giá nằm NGOÀI [P1, P99]
p1, p99 = np.percentile(gia_sach,1) , np.percentile(gia_sach,99)
so_ngoai = ((gia_sach < p1) | (gia_sach > p99)).sum()            # gợi ý: (gia_sach < p1) | (gia_sach > p99)

# --- Ô kiểm tra ---
assert round(p1) == 16369 and round(p99) == 686144
assert so_ngoai == 354
print(f"Khoảng [P1, P99] = [{p1:,.0f}, {p99:,.0f}] — {so_ngoai} giá nằm ngoài (~2%).")

Khoảng [P1, P99] = [16,369, 686,144] — 354 giá nằm ngoài (~2%).


P99 ≈ 686 nghìn CLP — mọi giá trên ngưỡng này (kể cả phòng 97 triệu của lab 2) là ứng
viên gắn cờ ở bài 10. Ngưỡng được xác định **từ dữ liệu** thay vì đặt tùy ý.

## Phần 2 · Ma trận review 4 năm × 12 tháng (~20 phút)

Bảng dưới là số review Santiago theo năm (2022–2025) và tháng (1–12),
trích từ 690 nghìn review của lab 2.

In [39]:
M = np.array([
    [ 3231,  2854,  3469,  3449,  3536,  3155,  4580,  4509,  4963,  4951,  5100,  4590],
    [ 5315,  4584,  5555,  5186,  5066,  5122,  6997,  7440,  6503,  7528,  8523,  7371],
    [ 8218,  6959,  9330,  9997,  8299,  9705, 13066, 13659, 11722, 12480, 15176, 12656],
    [14636, 11652, 16565, 14041, 14509, 15192, 21726, 22904, 17301, 20791, 23028, 19087],
])  # hàng = 2022, 2023, 2024, 2025; cột = tháng 1..12
nam = np.array([2022, 2023, 2024, 2025])

M.shape

(4, 12)

In [41]:
# TODO: dùng đúng axis để tính
tong_moi_nam = M.sum(axis= 1)        # tổng review từng năm (4 số)
tb_moi_thang = M.mean(axis=0)        # trung bình mỗi tháng, gộp 4 năm (12 số)

# --- Ô kiểm tra ---
assert list(tong_moi_nam) == [48387, 75190, 131267, 211432]
assert round(float(tb_moi_thang[0])) == 7850
print("Năm 2025 gấp ~4.4 lần năm 2022 — thị trường phục hồi rất nhanh sau COVID.")

Năm 2025 gấp ~4.4 lần năm 2022 — thị trường phục hồi rất nhanh sau COVID.


In [51]:
# TODO: argmax hai chiều
thang_dinh = M.argmax(axis = 1) + 1         # tháng cao nhất của TỪNG năm (4 số, tính theo 1..12 — nhớ +1)
nam_dinh_thang1 = nam[M[:,0].argmax()]     # NĂM có tháng-1 cao nhất (dùng nam[...])

# --- Ô kiểm tra ---
assert list(thang_dinh) == [11, 11, 11, 11]
assert nam_dinh_thang1 == 2025
print("Cả 4 năm đều đỉnh vào tháng 11 (theo đếm thô).")

Cả 4 năm đều đỉnh vào tháng 11 (theo đếm thô).


Câu hỏi: bảng đếm thô này trộn lẫn **hai tín hiệu** — mùa vụ trong năm và
đà tăng trưởng của cả thị trường. Muốn nhìn riêng "mùa vụ", hãy chuẩn hoá trong từng năm
(chia mỗi hàng cho trung bình của chính hàng đó) rồi xem tháng nào thật sự cao. Chọn cách
đo cũng là một quyết định phân tích — bài 8 sẽ quay lại chủ đề này.

## Phần 3 · Bài tự làm ✅ mở (làm sớm tại lớp hoặc làm tại nhà)

Bạn được dùng AI theo quy trình 5 bước; hãy ghi lại prompt chính và cách kiểm chứng.

### Tự làm 1 · Mười giá lớn nhất bằng argsort

Dùng `np.argsort` trên `gia_sach` để lấy **10 giá cao nhất** (không dùng pandas).
In ra và đối chiếu: giá cao nhất có đúng 97.000.045 như lab 2 không?

### Tự làm 2 · Bootstrap trung vị (nâng cao)

Ước lượng độ bất định của trung vị 59.000 CLP bằng `rng = np.random.default_rng(42)`:
lặp 1.000 lần việc *lấy mẫu lại có hoàn lại* (`rng.choice(gia_sach, size=gia_sach.size,
replace=True)`) và tính trung vị mỗi lần; lấy `np.percentile(..., [2.5, 97.5])` của
1.000 trung vị đó. Khoảng thu được nói lên điều gì? (Kỹ thuật này tên là bootstrap —
có thể dùng để mô tả độ bất định của một ước lượng trong báo cáo.)

In [64]:
# tự làm 1
top10 = gia_sach[np.argsort(gia_sach)[-10:]]
print("Top 10 giá cao nhất:", top10)

# tự làm 2
rng = np.random.default_rng(42)
# Tạo mảng chứa 1000 trung vị từ 1000 lần lấy mẫu
boot_medians = [np.median(rng.choice(gia_sach, size=gia_sach.size, replace=True)) for _ in range(1000)]

# Tính khoảng tin cậy 95% (từ phân vị 2.5 đến 97.5)
conf_interval = np.percentile(boot_medians, [2.5, 97.5])
print(f"Khoảng tin cậy 95% của trung vị: {conf_interval}")

Top 10 giá cao nhất: [10725138. 12448437. 21736032. 36514883. 36544657. 43741160. 45076009.
 48863551. 82566932. 97000045.]
Khoảng tin cậy 95% của trung vị: [58261.8125 59708.1125]


## Tóm tắt buổi lab

| Nội dung chính | Sẽ gặp lại ở |
|---|---|
| NaN: đếm bằng `isnan`, làm sạch chủ động | missing values (bài 10) |
| Mặt nạ bool: đếm / tỷ lệ / ghép `&` | lọc pandas (bài 4–5), tính KPI |
| np.where gắn nhãn; ranh giới là quyết định | gắn cờ QA (bài 10) |
| Phân vị làm ngưỡng ngoại lai từ dữ liệu | bộ quy tắc QA (bài 10) |
| axis & argmax trên ma trận thật | pivot_table (bài 5), resample (bài 8) |

Buổi lý thuyết tiếp theo: **pandas** — mảng có tên cột và nhiều thao tác NumPy vẫn áp dụng được.